# Topic 06 — Solutions: Merge / Join / Concat

*Reference solutions for the objective tasks. Try the practice first.* The Warm-Up concept questions, Interview Lens and Reflection have **no key** — by design.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
oi = items.merge(orders[['order_id','channel','customer_id']], on='order_id', how='left')
customers = pd.read_csv(RAW+'customers.csv', dtype={'customer_id':str}).drop_duplicates('customer_id')
products['list_price'] = np.where(products['list_price']<0, np.nan, products['list_price'])


### Warm-Up — NumPy recall (self-check)

In [ ]:
left = np.array([1,2,3,4,5]); right = np.array([3,4,5,6])
matched = np.intersect1d(left, right); orphans_n = int((~np.isin(left, right)).sum())
print(matched, orphans_n)

### Core lesson tasks

In [ ]:
m = items.merge(products[['product_id','unit_cost']], on='product_id', how='left', validate='many_to_one')
print(len(m))

In [ ]:
m['line_profit'] = m['line_revenue'] - m['quantity']*m['unit_cost']
print(m['unit_cost'].isna().sum())

In [ ]:
om = orders.merge(customers[['customer_id']], on='customer_id', how='left', indicator=True)
orphans = om[om['_merge']=='left_only']
print(orphans.shape[0])

### Mixed review (earlier topics)

In [ ]:
master = items.merge(products[['product_id','unit_cost']], on='product_id', how='left').merge(orders[['order_id','channel']], on='order_id', how='left')
master['line_profit'] = master['line_revenue'] - master['quantity']*master['unit_cost']
print(master.groupby('channel')['line_profit'].sum().sort_values(ascending=False))

In [ ]:
n_loss = (master['line_profit'] < 0).sum()
print(int(n_loss))

### Data detective

In [ ]:
master = items.merge(products[['product_id','unit_cost']], on='product_id', how='left', validate='many_to_one').merge(orders[['order_id','channel']], on='order_id', how='left', validate='many_to_one')
master['line_profit'] = master['line_revenue'] - master['quantity']*master['unit_cost']
print(master.groupby('channel')['line_profit'].sum())

### Interview Lens — discussion points (not full answers)
- **Q18:** inner=intersection; left=all orders (orphans→NaN); right=all customers; outer=union.
- **Q19:** duplicate join keys fan out rows and inflate sums. Detect with `validate=`, row-count asserts, `indicator=True`, `df[key].is_unique`.